# Part 1: Tokenization and Vocabulary

## Notebook 2 — BPE from Scratch

We train a minimal Byte-Pair Encoding (BPE) tokenizer to understand:
- How BPE learns merge rules from data
- How vocabulary size affects compression
- Why the vocabulary is dataset-dependent


### 1. Create a toy robotics corpus

BPE learns from data. We create a small corpus of robot task descriptions to see how subword patterns emerge.


In [1]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

corpus = [
    "pick up the red cube",
    "place the cube on the table",
    "move the robotic arm to position x",
    "grasp the object with the gripper",
    "rotate the end effector by 90 degrees",
    "open the gripper to release the object",
    "move to the home position",
    "pick and place the blue block",
    "stack the red cube on the blue cube",
    "push the object forward",
]
print(f"Corpus size: {len(corpus)} sentences")
print(f"Sample: {corpus[0]}")


Corpus size: 10 sentences
Sample: pick up the red cube


### 2. Train a BPE tokenizer

Byte-Pair Encoding works by starting with individual characters and repeatedly merging the most frequent pair of adjacent tokens:

1. Start with every character in the corpus as its own token
2. Count how often each adjacent pair appears
3. Merge the most frequent pair into a new token
4. Repeat until the vocabulary reaches the target size

The name comes from replacing the most common "byte pair" with a single token. In the next cell we train with a small vocabulary (100 tokens) so the merges are easy to inspect. GPT-2 uses 50k tokens.


In [2]:
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

trainer = trainers.BpeTrainer(
    vocab_size=100,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
    min_frequency=1,
)

tokenizer.train_from_iterator(corpus, trainer)
print(f"Vocab size: {tokenizer.get_vocab_size()}")





Vocab size: 100


In [ ]:
# Show the merge rules BPE learned (most frequent pairs first)
merges = sorted(tokenizer.model.merges, key=tokenizer.model.merges.get)
print(f"Learned {len(merges)} merge rules. First 15:")
for i, (a, b) in enumerate(merges[:15]):
    print(f"  {i+1:2d}. '{a}' + '{b}' → '{a}{b}'")

### 3. Inspect the vocabulary

The vocabulary shows how BPE builds a hierarchy: characters → common subwords → frequent words.


In [3]:
vocab = tokenizer.get_vocab()
# Show first 30 tokens
for token, tid in sorted(vocab.items(), key=lambda x: x[1])[:30]:
    print(f"  {tid:3d}: {repr(token)}")


    0: '<pad>'
    1: '<unk>'
    2: '<bos>'
    3: '<eos>'
    4: '0'
    5: '9'
    6: 'a'
    7: 'b'
    8: 'c'
    9: 'd'
   10: 'e'
   11: 'f'
   12: 'g'
   13: 'h'
   14: 'i'
   15: 'j'
   16: 'k'
   17: 'l'
   18: 'm'
   19: 'n'
   20: 'o'
   21: 'p'
   22: 'r'
   23: 's'
   24: 't'
   25: 'u'
   26: 'v'
   27: 'w'
   28: 'x'
   29: 'y'


### 4. Tokenize with the trained tokenizer

Compare our small vocabulary tokenizer with GPT-2's 50k vocabulary on the same sentence.


In [4]:
sentence = "pick up the red cube and place it on the table"
output = tokenizer.encode(sentence)
print(f"Sentence: {sentence}")
print(f"Token IDs: {output.ids}")
print(f"Tokens:   {output.tokens}")


Sentence: pick up the red cube and place it on the table
Token IDs: [60, 98, 31, 68, 41, 75, 71, 14, 24, 40, 31, 45, 33, 10]
Tokens:   ['pick', 'up', 'the', 'red', 'cube', 'and', 'place', 'i', 't', 'on', 'the', 'ta', 'bl', 'e']


### 5. Effect of vocabulary size

Smaller vocab = fewer tokens to learn but longer sequences. Larger vocab = more tokens to learn but shorter sequences. This is a fundamental trade-off in tokenization.


In [11]:
# Train tokenizers with different vocab sizes
for vocab_size in [50, 100, 200]:
    t = Tokenizer(models.BPE())
    t.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
        min_frequency=1,
    )
    t.train_from_iterator(corpus, trainer)
    output = t.encode(sentence)
    print(f"Vocab size {vocab_size:3d}: {len(output.ids):2d} tokens -> {output.tokens}")




Vocab size  50: 23 tokens -> ['p', 'i', 'ck', 'u', 'p', 'the', 're', 'd', 'cube', 'a', 'n', 'd', 'p', 'l', 'ac', 'e', 'i', 't', 'on', 'the', 'ta', 'bl', 'e']




Vocab size 100: 14 tokens -> ['pick', 'up', 'the', 'red', 'cube', 'and', 'place', 'i', 't', 'on', 'the', 'ta', 'bl', 'e']


Vocab size 200: 12 tokens -> ['pick', 'up', 'the', 'red', 'cube', 'and', 'place', 'i', 't', 'on', 'the', 'table']



### The Gist

BPE learns compression rules from data. The vocabulary size controls the compression/sequence-length trade-off. In Part 3, we'll see how FAST action tokenization applies BPE to Discrete Cosine Transform coefficients instead of text characters.
